# Convert Arivis segmentation exports to LightSuite Sample Space v1

This notebook converts **Arivis Vision4D / Blob Finder** exports into formats accepted by
`lightsuite brain import-annotations`:

| LightSuite format | Arivis source | Use when |
|-------------------|---------------|----------|
| `points_csv` | `*-features.xlsx` or `*-features.csv` (COM columns) | Cell / blob centroids (fast, small) |
| `mask_tiff` | `MASK/*.tiff` (per-slice) **or** one multi-page `*-objects.tiff` | Full segmentation volume |

Annotations must match the **native grid of the stack you segmented on** (not the 20 µm
registration preview). Use `sample_reference.json` from `lightsuite brain preprocess` when
available, or set `REFERENCE_OVERRIDE` in section 0 (e.g. mesoSPIM 2.5× ROI before multires).

### Marianna CMU (2.5× ROI → 0.8× overview → Perens)

1. **This notebook** — convert Arivis exports at **2.5× native** (`REFERENCE_OVERRIDE`).
2. **`lightsuite multires register`** — standard overview ↔ ROI registration (`marianna_multires.yaml`).
3. **Manual coordinate transfer** — warp 2.5× `points.csv` / `mask.tif` into **0.8× native** indices (not automated yet).
4. **`lightsuite brain import-annotations`** — warp into Perens atlas space (`marianna_yosi_parity.yaml`, `save_path: register_Yosi_test_python`).
5. **`lightsuite brain export --write-csv`** — per-region intensity tables in the Perens ontology.

See also: [Annotation import](../docs/annotation_import.md), [Multiresolution usage](../docs/usage_multiresolution.md).

## 0. Configure paths

Set `SAMPLE` to a preset or choose `custom` and edit the paths manually.

| Preset | Segmented stack | Brain `save_path` (0.8× atlas registration) |
|--------|-----------------|-----------------------------------------------|
| `julie_buron` | 1× mesoSPIM overview | `registered_allen` |
| `marianna_2p5x` | 2.5× mesoSPIM ROI | `register_Yosi_test_python` (Perens) |

In [11]:
from pathlib import Path

SAMPLE = "marianna_2p5x"  # "julie_buron" | "marianna_2p5x" | "custom"

if SAMPLE == "julie_buron":
    BRAIN_SAVE_PATH = Path("/media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/registered_allen")
    ARIVIS_EXPORT = Path(
        "/media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/FromArivis"
    )
    OUTPUT_DIR = Path(
        "/media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/converted"
    )
    LABEL = "arivis_561"
    REFERENCE_OVERRIDE = None  # use sample_reference.json from preprocess
elif SAMPLE == "marianna_2p5x":
    BRAIN_SAVE_PATH = Path(
        "/media/gbm/NVME2/ALICe-pipelines-data/Marianna_CMU/MATLAB_python_comparison/register_Yosi_test_python"
    )
    ARIVIS_EXPORT = Path(
        "/media/gbm/NVME2/ALICe-pipelines-data/Marianna_CMU/segmentation/OUTPUT_ARIVIS"
    )
    OUTPUT_DIR = Path(
        "/media/gbm/NVME2/ALICe-pipelines-data/Marianna_CMU/segmentation/converted"
    )
    LABEL = "arivis_2p5x"
    # Segmentation was on the 2.5× ROI stack (not the 0.8× overview used for atlas registration).
    REFERENCE_OVERRIDE = {
        "format": "lightsuite_sample_space_v1",
        "sample_name": "Marianna",
        "shape_yxz": [2048, 2048, 1234],
        "voxel_um": [2.6, 2.6, 3.0],
        "index_base": 1,
        "axis_order": "xyz",
        "coordinate_units": "voxel_indices",
        "orientation_applied": False,
    }
else:
    BRAIN_SAVE_PATH = Path("/path/to/brain/save_path")
    ARIVIS_EXPORT = Path("/path/to/arivis/export")
    OUTPUT_DIR = Path("/path/to/converted")
    LABEL = "arivis"
    REFERENCE_OVERRIDE = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("sample preset:", SAMPLE)
print("brain save_path (atlas QC):", BRAIN_SAVE_PATH)
print("arivis export:", ARIVIS_EXPORT)
print("output:", OUTPUT_DIR)

save_path: /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/registered_allen
arivis export: /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/FromArivis
output: /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/converted


## 1. Load the LightSuite native grid contract

After `preprocess`, LightSuite writes `sample_reference.json`. Every imported annotation must
align with this grid:

- **Shape** `(Y, X, Z)` = `shape_yxz` → `(ny, nx, nz)`
- **Voxel size** `[x, y, z]` in µm
- **Indices** are **1-based** — the first voxel center is `(1, 1, 1)`, not `(0, 0, 0)`
- **Axis order** for point CSVs is always `x, y, z`
- Do **not** apply `registration.orientation` yourself; import applies it during the warp

In [12]:
import json

if REFERENCE_OVERRIDE is not None:
    reference = REFERENCE_OVERRIDE
    print("Using REFERENCE_OVERRIDE (segmentation native grid)")
else:
    ref_path = BRAIN_SAVE_PATH / "sample_reference.json"
    if not ref_path.is_file():
        raise FileNotFoundError(
            f"Missing {ref_path}. Run: uv run lightsuite brain preprocess -c your_config.yaml "
            "or set REFERENCE_OVERRIDE in section 0."
        )
    reference = json.loads(ref_path.read_text(encoding="utf-8"))

ny, nx, nz = reference["shape_yxz"]
voxel_um = reference["voxel_um"]

print(json.dumps(reference, indent=2))
print()
print(f"Expected mask shape (Y, X, Z): ({ny}, {nx}, {nz})")
print(f"Point bounds: 1 <= x <= {nx}, 1 <= y <= {ny}, 1 <= z <= {nz}")

{
  "format": "lightsuite_sample_space_v1",
  "sample_name": "Julie",
  "shape_yxz": [
    2048,
    2048,
    1361
  ],
  "voxel_um": [
    6.55,
    6.55,
    5.0
  ],
  "index_base": 1,
  "axis_order": "xyz",
  "coordinate_units": "voxel_indices",
  "orientation_applied": false
}

Expected mask shape (Y, X, Z): (2048, 2048, 1361)
Point bounds: 1 <= x <= 2048, 1 <= y <= 2048, 1 <= z <= 1361


## 2. Inspect the Arivis export

LightSuite supports two common **Blob Finder** export layouts:

**Layout A — per-slice masks + XLSX** (Julie Buron example):

```
FromArivis/
├── 1_2ch_1X-features.xlsx
└── MASK/
    ├── …_Z00000.tiff
    └── …
```

**Layout B — single mask stack + CSV** (Marianna 2.5× example):

```
OUTPUT_ARIVIS/
├── U87_DJ_1week_2.5x-features.csv
└── MASK_U87_DJ_1week_2-objects.tiff   # multi-page (Z, Y, X)
```

**Coordinate convention:** Arivis COM columns in **pixels** are **0-based** (first pixel = 0).
LightSuite requires **+1** on each axis.

In [13]:
mask_dir = ARIVIS_EXPORT / "MASK"
xlsx_files = sorted(ARIVIS_EXPORT.glob("*-features.xlsx"))
csv_files = sorted(ARIVIS_EXPORT.glob("*-features.csv"))
mask_slice_files = sorted(mask_dir.glob("*.tif*")) if mask_dir.is_dir() else []
mask_stack_files = sorted(ARIVIS_EXPORT.glob("*-objects.tiff")) + sorted(
    ARIVIS_EXPORT.glob("*-objects.tif")
)

feature_files = xlsx_files or csv_files
features_format = "xlsx" if xlsx_files else "csv" if csv_files else None
mask_mode = (
    "slices"
    if mask_slice_files
    else "stack"
    if mask_stack_files
    else None
)

print("Features:", [p.name for p in feature_files], f"({features_format})")
print("Mask mode:", mask_mode)
if mask_mode == "slices":
    print("Mask slice count:", len(mask_slice_files))
    print("First mask:", mask_slice_files[0].name)
    print("Last mask:", mask_slice_files[-1].name)
elif mask_mode == "stack":
    print("Mask stack:", mask_stack_files[0].name)

if not feature_files:
    raise FileNotFoundError(f"No *-features.xlsx or *-features.csv in {ARIVIS_EXPORT}")

Features spreadsheets: ['1_2ch_1X-features.xlsx']
Mask slice count: 1361
First mask: 1_2ch_1X-objects_T00000_Z00000.tiff
Last mask: 1_2ch_1X-objects_T00000_Z01360.tiff


## 3. Read Arivis features (XLSX or CSV)

XLSX exports often use non-standard styling that breaks `openpyxl` / `pandas.read_excel`.
We parse the worksheet XML directly from the XLSX zip archive (data only, no styles).

CSV exports (Marianna) are read with the standard library `csv` module.

Look for columns like:
- `X (px), Center of Mass (Intensities) #1`
- `Y (px), Center of Mass (Intensities) #1`
- `Z (px), Center of Mass (Intensities) #1`

In [14]:
import csv
import zipfile
import xml.etree.ElementTree as ET

NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"


def _cell_text(cell) -> str:
    is_elem = cell.find(f"{NS}is")
    if is_elem is not None:
        t = is_elem.find(f"{NS}t")
        return t.text if t is not None else ""
    v = cell.find(f"{NS}v")
    return v.text if v is not None else ""


def read_arivis_xlsx(path: Path) -> tuple[list[str], list[list[str]]]:
    """Return (header_row, data_rows) from first worksheet."""
    with zipfile.ZipFile(path) as zf:
        sheet_name = "xl/worksheets/sheet.xml"
        if sheet_name not in zf.namelist():
            sheet_name = "xl/worksheets/sheet1.xml"
        root = ET.fromstring(zf.read(sheet_name))

    rows: list[list[str]] = []
    for row in root.findall(f"{NS}sheetData/{NS}row"):
        vals = [_cell_text(c) for c in row.findall(f"{NS}c")]
        rows.append(vals)

    if not rows:
        raise ValueError(f"No rows in {path}")
    return rows[0], rows[1:]


def read_arivis_csv(path: Path) -> tuple[list[str], list[list[str]]]:
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        rows = list(reader)
    if not rows:
        raise ValueError(f"No rows in {path}")
    return rows[0], rows[1:]


FEATURES_PATH = feature_files[0]
if features_format == "xlsx":
    header, data_rows = read_arivis_xlsx(FEATURES_PATH)
else:
    header, data_rows = read_arivis_csv(FEATURES_PATH)

print(f"Loaded {FEATURES_PATH.name}: {len(data_rows)} table rows")
print("Columns:")
for i, col in enumerate(header):
    print(f"  [{i}] {col}")

Loaded 1_2ch_1X-features.xlsx: 9105 table rows
Columns:
  [0] Type
  [1] Name
  [2] Volume, Volume (µm³)
  [3] VoxelCount, Volume
  [4] RelativeVolume, Volume
  [5] X (px), Center of Mass (Intensities) #1
  [6] Y (px), Center of Mass (Intensities) #1
  [7] Z (px), Center of Mass (Intensities) #1


In [15]:
def find_com_columns(header: list[str]) -> tuple[int, int, int]:
    """Map x, y, z column indices from Arivis header names."""
    lowered = [h.lower() for h in header]

    def pick(axis: str) -> int:
        for i, h in enumerate(lowered):
            if axis in h and "center of mass" in h:
                return i
        for i, h in enumerate(lowered):
            if h.startswith(f"{axis} (px)"):
                return i
        raise KeyError(f"Could not find {axis} COM column in header: {header}")

    return pick("x"), pick("y"), pick("z")


ix, iy, iz = find_com_columns(header)
print(f"COM columns: x=[{ix}] {header[ix]}, y=[{iy}] {header[iy]}, z=[{iz}] {header[iz]}")

COM columns: x=[5] X (px), Center of Mass (Intensities) #1, y=[6] Y (px), Center of Mass (Intensities) #1, z=[7] Z (px), Center of Mass (Intensities) #1


## 4. Convert segments → `points.csv`

LightSuite expects a CSV with header columns **`x`**, **`y`**, **`z`** (1-based native voxel indices).
Extra **numeric** columns (volume, voxel count, intensity, …) are kept as optional features.
Text columns such as segment `Name` are skipped automatically at import.

In [16]:
import csv
import numpy as np

POINTS_CSV = OUTPUT_DIR / f"{LABEL}_points.csv"

# Optional extra columns to preserve (adjust if your export differs)
EXTRA_COLS = []
for name in ("Name", "Volume, Volume (µm³)", "VoxelCount, Volume"):
    if name in header:
        EXTRA_COLS.append(name)

type_col = header.index("Type") if "Type" in header else None

points_xyz = []
with POINTS_CSV.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["x", "y", "z"] + EXTRA_COLS)

    for row in data_rows:
        if type_col is not None and row[type_col] != "Segment":
            continue
        if max(ix, iy, iz) >= len(row):
            continue
        try:
            x = float(row[ix]) + 1.0  # Arivis 0-based → LightSuite 1-based
            y = float(row[iy]) + 1.0
            z = float(row[iz]) + 1.0
        except (TypeError, ValueError):
            continue
        extras = [row[header.index(c)] if c in header else "" for c in EXTRA_COLS]
        writer.writerow([x, y, z] + extras)
        points_xyz.append([x, y, z])

points_xyz = np.asarray(points_xyz, dtype=np.float64)
print(f"Wrote {len(points_xyz)} points → {POINTS_CSV}")
if len(points_xyz):
    print(
        "Ranges (1-based):",
        f"x [{points_xyz[:,0].min():.1f}, {points_xyz[:,0].max():.1f}]",
        f"y [{points_xyz[:,1].min():.1f}, {points_xyz[:,1].max():.1f}]",
        f"z [{points_xyz[:,2].min():.1f}, {points_xyz[:,2].max():.1f}]",
    )

Wrote 9105 points → /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/converted/arivis_561_points.csv
Ranges (1-based): x [190.5, 1800.3] y [18.4, 2048.0] z [1.2, 1359.8]


## 5. Validate points against `sample_reference.json`

Points outside `1 … nx/ny/nz` are dropped at import. Ideally everything should already be in range.

In [17]:
if len(points_xyz):
    in_bounds = (
        (points_xyz[:, 0] >= 1) & (points_xyz[:, 0] <= nx)
        & (points_xyz[:, 1] >= 1) & (points_xyz[:, 1] <= ny)
        & (points_xyz[:, 2] >= 1) & (points_xyz[:, 2] <= nz)
    )
    n_ok = int(in_bounds.sum())
    n_bad = len(points_xyz) - n_ok
    print(f"In bounds: {n_ok}/{len(points_xyz)} ({100 * n_ok / len(points_xyz):.1f}%)")
    if n_bad:
        bad = points_xyz[~in_bounds]
        print(f"WARNING: {n_bad} points out of bounds — will be dropped at import")
        print("First few bad points:", bad[:5])
else:
    print("No points to validate")

In bounds: 9105/9105 (100.0%)


## 6. Build `mask.tif` (optional)

Skip this section if you only need point centroids.

LightSuite expects one 3D binary TIFF:

- Shape `(Y, X, Z)` matching `shape_yxz`
- Values `0` / `255` (or `0` / `1`)
- Z-stack layout: one 2D page per Z slice (we write `(Z, Y, X)` pages)

**Layout A:** stack per-slice masks from `MASK/`. **Layout B:** copy or binarize a single
multi-page `*-objects.tiff` export (Marianna).

**Note:** a full 2048×2048×1234 mask is ~5 GB. Ensure enough disk space and RAM.

In [18]:
import re

import numpy as np
import tifffile

BUILD_MASK = True  # set False to skip mask build
MASK_TIFF = OUTPUT_DIR / f"{LABEL}_mask.tif"

if not BUILD_MASK:
    print("Skipping mask build (BUILD_MASK=False)")
elif mask_mode is None:
    print("No mask export found — skipping")
elif mask_mode == "stack":
    source_mask = mask_stack_files[0]
    stack_zyx = np.asarray(tifffile.imread(source_mask))
    if stack_zyx.ndim != 3:
        raise ValueError(f"Expected 3D mask stack in {source_mask}, got {stack_zyx.shape}")
    # Binarize label/object masks (0 / 255) for LightSuite import.
    stack_zyx = (stack_zyx > 0).astype(np.uint8) * 255
    print(f"Read mask stack {source_mask.name}, shape (Z, Y, X): {stack_zyx.shape}")
    print("Nonzero voxels:", int(np.count_nonzero(stack_zyx)))
    yxz_shape = (stack_zyx.shape[1], stack_zyx.shape[2], stack_zyx.shape[0])
    if yxz_shape != (ny, nx, nz):
        print(
            f"WARNING: mask shape {yxz_shape} != reference ({ny}, {nx}, {nz}). "
            "Import will fail unless shapes match."
        )
    tifffile.imwrite(MASK_TIFF, stack_zyx, photometric="minisblack")
    print(f"Wrote mask → {MASK_TIFF}")
else:
    z_pattern = re.compile(r"_Z(\d+)$")

    def z_index(path: Path) -> int:
        m = z_pattern.search(path.stem)
        if m is None:
            raise ValueError(f"Cannot parse Z index from {path.name}")
        return int(m.group(1))

    sorted_masks = sorted(mask_slice_files, key=z_index)
    z_indices = [z_index(p) for p in sorted_masks]
    print(f"Stacking {len(sorted_masks)} slices, Z index {z_indices[0]} … {z_indices[-1]}")

    if z_indices[-1] - z_indices[0] + 1 != len(sorted_masks):
        print("WARNING: Z indices may have gaps — check Arivis export completeness")

    planes = []
    for i, path in enumerate(sorted_masks):
        plane = tifffile.imread(path)
        if plane.ndim != 2:
            raise ValueError(f"Expected 2D mask slice in {path}, got shape {plane.shape}")
        planes.append(plane)
        if i % 200 == 0:
            print(f"  read {i}/{len(sorted_masks)} …")

    stack_zyx = np.stack(planes, axis=0)
    print("Stack shape (Z, Y, X):", stack_zyx.shape)
    print("Nonzero voxels:", int(np.count_nonzero(stack_zyx)))

    yxz_shape = (stack_zyx.shape[1], stack_zyx.shape[2], stack_zyx.shape[0])
    if yxz_shape != (ny, nx, nz):
        print(
            f"WARNING: mask shape {yxz_shape} != reference ({ny}, {nx}, {nz}). "
            "Import will fail unless shapes match."
        )

    tifffile.imwrite(MASK_TIFF, stack_zyx, photometric="minisblack")
    print(f"Wrote mask → {MASK_TIFF}")

Stacking 1361 slices, Z index 0 … 1360
  read 0/1361 …
  read 200/1361 …
  read 400/1361 …
  read 600/1361 …
  read 800/1361 …
  read 1000/1361 …
  read 1200/1361 …
Stack shape (Z, Y, X): (1361, 2048, 2048)
Nonzero voxels: 773309
Wrote mask → /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/converted/arivis_561_mask.tif


## 7. Quick visual QC (optional)

Confirm points and/or mask alignment with the native acquisition grid before wiring
`import-annotations`.

Set **`QC_MODE`** in the next cell:

| Mode | Use when | Extra setup |
|------|----------|-------------|
| **`matplotlib`** | Low RAM, headless runs, quick static check | Default dev env (matplotlib) |
| **`napari`** | Interactive 3D slice browsing; GPU can speed rendering | `uv sync --extra gui` |

**Napari profiles** (`NAPARI_PROFILE`, only when `QC_MODE == "napari"`):

| Profile | RAM | What you see |
|---------|-----|--------------|
| **`light`** | Low | 20 µm registration preview + spots rescaled to that grid |
| **`native_lazy`** | Medium | Full-res mask (lazy/dask) + native spots |
| **`native_full`** | High | Full-res mask loaded entirely + native spots |

Pick **`matplotlib`** or **`napari` + `light`** on memory-constrained machines. Use
**`native_full`** only if you have enough RAM for the full native mask stack.

In [22]:
# --- Choose QC workflow ---
QC_MODE = "napari"  # "matplotlib" | "napari"

# Napari-only (ignored when QC_MODE == "matplotlib")
NAPARI_PROFILE = "light"  # "light" | "native_lazy" | "native_full"
NAPARI_REG_CHANNEL = 1  # chan_* used for the 20 µm preview background


def sample_xyz_to_napari_zyx(coords_xyz_1based: np.ndarray) -> np.ndarray:
    """Map 1-based sample (x, y, z) to Napari (z, y, x) indices."""
    pts = np.asarray(coords_xyz_1based, dtype=np.float64)
    if pts.size == 0:
        return np.zeros((0, 3), dtype=np.float64)
    return np.column_stack([pts[:, 2] - 1.0, pts[:, 1] - 1.0, pts[:, 0] - 1.0])


def native_xyz_to_preview_zyx(
    coords_xyz_1based: np.ndarray,
    *,
    native_shape_yxz: tuple[int, int, int],
    preview_shape_zyx: tuple[int, int, int],
) -> np.ndarray:
    """Rescale native 1-based xyz onto a downsampled preview grid for overlay."""
    ny_n, nx_n, nz_n = (int(v) for v in native_shape_yxz)
    pz, py, px = (int(v) for v in preview_shape_zyx)
    sx = (px - 1) / max(nx_n - 1, 1)
    sy = (py - 1) / max(ny_n - 1, 1)
    sz = (pz - 1) / max(nz_n - 1, 1)
    x = coords_xyz_1based[:, 0] - 1.0
    y = coords_xyz_1based[:, 1] - 1.0
    z = coords_xyz_1based[:, 2] - 1.0
    return np.column_stack([z * sz, y * sy, x * sx])


def _contrast_limits(volume: np.ndarray) -> tuple[float, float]:
    positive = volume[volume > 0]
    if positive.size:
        lo, hi = np.percentile(positive, (1.0, 99.5))
        return float(lo), float(hi)
    return float(volume.min()), float(volume.max())


def _load_tiff_zyx_for_napari(path: Path) -> np.ndarray:
    """Load a TIFF as Napari (Z, Y, X), stacking per-page IFDs when needed.

    ``tifffile.imread`` only returns the first plane for LightSuite registration
    preview TIFFs (one IFD per Z). Those must be stacked explicitly.
    """
    path = path.expanduser()
    data = np.asarray(tifffile.imread(path))
    if data.ndim == 3:
        return data
    if data.ndim != 2:
        msg = f"Unsupported TIFF shape {data.shape} in {path}"
        raise ValueError(msg)
    with tifffile.TiffFile(path) as tif:
        if len(tif.pages) <= 1:
            return data[np.newaxis, ...]
        planes = [np.asarray(page.asarray()) for page in tif.pages]
        return np.stack(planes, axis=0)


z_show = nz // 2
z0 = z_show - 1

if QC_MODE == "matplotlib":
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    if BUILD_MASK and MASK_TIFF.is_file():
        with tifffile.TiffFile(MASK_TIFF) as tif:
            mask_slice = tif.pages[z0].asarray()
        axes[0].imshow(mask_slice, cmap="gray")
        axes[0].set_title(f"Arivis mask slice z={z_show}")
    else:
        axes[0].set_title("No mask built")
        axes[0].axis("off")

    if len(points_xyz):
        sel = np.abs(points_xyz[:, 2] - z_show) <= 2.0
        sub = points_xyz[sel]
        axes[1].scatter(sub[:, 0], sub[:, 1], s=4, c="red", alpha=0.5)
        axes[1].set_xlim(1, nx)
        axes[1].set_ylim(ny, 1)
        axes[1].set_title(f"Points near z={z_show} ({int(sel.sum())} cells)")
    else:
        axes[1].set_title("No points")

    for ax in axes:
        ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

elif QC_MODE == "napari":
    try:
        import napari
    except ImportError as exc:
        raise ImportError(
            "Napari QC requires GUI extras: uv sync --extra gui"
        ) from exc

    viewer = napari.Viewer(title=f"LightSuite import QC — {LABEL}")
    napari_pts: np.ndarray | None = None

    if NAPARI_PROFILE == "light":
        regopts_path = BRAIN_SAVE_PATH / "regopts.json"
        if not regopts_path.is_file():
            raise FileNotFoundError(
                f"Missing {regopts_path}. Run preprocess first for reg-preview QC."
            )
        regopts = json.loads(regopts_path.read_text(encoding="utf-8"))
        registres_um = float(regopts["registres_um"])
        preview_path = BRAIN_SAVE_PATH / f"chan_{NAPARI_REG_CHANNEL}_sample_register_{int(registres_um)}um.tif"
        if not preview_path.is_file():
            raise FileNotFoundError(f"Missing registration preview: {preview_path}")

        preview_zyx = _load_tiff_zyx_for_napari(preview_path)
        clow, chigh = _contrast_limits(preview_zyx)
        viewer.add_image(
            preview_zyx,
            name=f"reg preview ch{NAPARI_REG_CHANNEL}",
            colormap="gray",
            contrast_limits=(clow, chigh),
            opacity=1.0,
        )
        if len(points_xyz):
            napari_pts = native_xyz_to_preview_zyx(
                points_xyz,
                native_shape_yxz=(ny, nx, nz),
                preview_shape_zyx=tuple(int(v) for v in preview_zyx.shape),
            )
        print(
            f"Napari light profile: {preview_path.name} "
            f"shape {tuple(int(v) for v in preview_zyx.shape)} (Z, Y, X)"
        )

    elif NAPARI_PROFILE in {"native_lazy", "native_full"}:
        if BUILD_MASK and MASK_TIFF.is_file():
            if NAPARI_PROFILE == "native_lazy":
                import dask.array as da

                with tifffile.TiffFile(MASK_TIFF) as tif:
                    mask_zyx = da.from_zarr(tif.aszarr())
            else:
                mask_zyx = _load_tiff_zyx_for_napari(MASK_TIFF)
            viewer.add_image(
                mask_zyx,
                name="segmentation mask",
                colormap="green",
                opacity=0.35,
                blending="additive",
            )
        elif BUILD_MASK:
            print("No mask file — showing points only")
        if len(points_xyz):
            napari_pts = sample_xyz_to_napari_zyx(points_xyz)

    else:
        raise ValueError(
            f"Unknown NAPARI_PROFILE={NAPARI_PROFILE!r}; "
            "use 'light', 'native_lazy', or 'native_full'"
        )

    if napari_pts is not None and napari_pts.shape[0]:
        viewer.add_points(
            napari_pts,
            name="cells",
            size=3,
            face_color="red",
            symbol="disc",
        )
        print(f"Showing {napari_pts.shape[0]} points")
    else:
        print("No points to display")

    napari.run()

else:
    raise ValueError(f"Unknown QC_MODE={QC_MODE!r}; use 'matplotlib' or 'napari'")

Napari light profile: chan_1_sample_register_20um.tif shape (341, 671, 671) (Z, Y, X)
Showing 9105 points


## 8. YAML snippet for `import-annotations`

The snippet below is written to `OUTPUT_DIR/import_snippet.yaml`.

**Julie Buron (1×):** add to your brain config and run after `lightsuite brain register`.

**Marianna (2.5× → 0.8×):** paths in this snippet are the **2.5× native** conversion outputs.
Before import into `register_Yosi_test_python`, warp them to **0.8× native** indices (manual step),
then point `import.annotations` at the warped files and use
`examples/config/mesoSPIM/marianna_yosi_parity.yaml`:

```bash
uv run lightsuite brain import-annotations -c examples/config/mesoSPIM/marianna_yosi_parity.yaml
uv run lightsuite brain export --write-csv          -c examples/config/mesoSPIM/marianna_yosi_parity.yaml
```

In [20]:
import yaml

import_block = {
    "write_csv": True,
    "annotations": [
        {
            "format": "points_csv",
            "path": str(POINTS_CSV.resolve()),
            "label": f"{LABEL}_cells",
        },
    ],
}

if BUILD_MASK and MASK_TIFF.is_file():
    import_block["annotations"].append(
        {
            "format": "mask_tiff",
            "path": str(MASK_TIFF.resolve()),
            "label": f"{LABEL}_mask",
        }
    )

snippet_path = OUTPUT_DIR / "import_snippet.yaml"
snippet_path.write_text(yaml.dump({'import': import_block}, sort_keys=False), encoding="utf-8")
print("Copy into your config YAML:")
print()
print(yaml.dump({'import': import_block}, sort_keys=False))

Copy into your config YAML:

import:
  write_csv: true
  annotations:
  - format: points_csv
    path: /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/converted/arivis_561_points.csv
    label: arivis_561_cells
  - format: mask_tiff
    path: /media/gbm/NVME2/ALICe-pipelines-data/JulieBuron/SegmentationImports/converted/arivis_561_mask.tif
    label: arivis_561_mask



## 9. Test load with LightSuite adapters (optional)

Runs the same loaders used by `import-annotations` — useful to catch format errors before the full warp.

In [21]:
from lightsuite.config.models import AnnotationFormat, AnnotationImportConfig
from lightsuite.import_.adapters import load_mask_tiff, load_points_csv, prepare_points_for_sample
from lightsuite.import_.sample_reference import SampleReference, validate_mask_against_reference

ref = SampleReference.load(ref_path)

pts_spec = AnnotationImportConfig(format=AnnotationFormat.POINTS_CSV, path=POINTS_CSV)
loaded_pts = load_points_csv(pts_spec)
prepared = prepare_points_for_sample(loaded_pts, reference=ref)
print(
    f"Points: {loaded_pts.coordinates.shape[0]} loaded, "
    f"{prepared.coordinates.shape[0]} in bounds, "
    f"{prepared.metadata.get('n_dropped', 0)} dropped"
)

if BUILD_MASK and MASK_TIFF.is_file():
    mask_spec = AnnotationImportConfig(format=AnnotationFormat.MASK_TIFF, path=MASK_TIFF)
    loaded_mask = load_mask_tiff(mask_spec)
    validate_mask_against_reference(loaded_mask.volume.shape, ref.voxel_um, ref)
    print(f"Mask: shape {loaded_mask.volume.shape}, foreground voxels {loaded_mask.volume.sum()}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Points: 9105 loaded, 9105 in bounds, 0 dropped
Mask: shape (2048, 2048, 1361), foreground voxels 773309


## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Many points dropped | 0-based Arivis coords without `+1` | Re-run step 4 |
| Mask shape mismatch | Wrong Z count or segmented different stack | Re-export from Arivis on native grid |
| `openpyxl` errors on XLSX | Arivis styling | Use the XML reader in step 3 |
| Empty atlas mask after import | Registration / orientation issue | Re-run orientation + match-points |
| Segmented 20 µm preview TIFF | Coordinates not native resolution | Segment on full-res stack or rescale |
| Napari QC `ImportError` | GUI extras not installed | `uv sync --extra gui` |
| Napari hangs / OOM | Full native mask on large stack | Use `QC_MODE="matplotlib"` or `NAPARI_PROFILE="light"` |

**Next:** complete `register`, then `uv run lightsuite brain import-annotations -c your_config.yaml`.